# DenguePharma Analytics — pesquisa reproduzível

Pergunta clínica: em pacientes com dengue estabelecida, quais intervenções apresentam benefício, alívio sintomático ou sinais de dano?

Este notebook descreve **um corpus piloto por execução**. Metadados coletados ainda podem conter registros inelegíveis. As conclusões clínicas dependem de seleção e resultados revisados. Execute com o Python de `analytics/.venv` ou com o script documentado. Nenhuma coleta, curadoria ou publicação é realizada aqui.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image
from denguepharma_analytics.storage import default_root, require_published, public_summary, query_table, run_path

run_id = os.environ.get("DENGUEPHARMA_RUN_ID", "7a1897915598231a92e7b969185fc13d33e23d88e245d5c015bab26b13984424")
root = default_root()
manifest = require_published(root, run_id)
summary = public_summary(root, run_id)
folder = run_path(root, run_id)
display(pd.DataFrame([{
    "run_id": run_id, "input_hash": manifest["input_hash"],
    "code_hash": manifest["code_hash"], "version": manifest["version"],
    "protocol": str(manifest["protocol"]),
}]))

# Leitura paginada pela mesma interface pública usada pela aplicação.
# Nenhuma transformação científica é reimplementada no notebook.
def table(name):
    first = query_table(root, run_id, name, limit=1000)
    rows = list(first["rows"])
    for offset in range(1000, first["total"], 1000):
        rows.extend(query_table(root, run_id, name, offset=offset, limit=1000)["rows"])
    assert len(rows) == first["total"]
    columns = [field["name"] for field in manifest["tables"][name]["schema"]]
    return pd.DataFrame(rows, columns=columns)

def show(name):
    display(Markdown(f"**Tabela: {name} — {manifest['tables'][name]['rows']} linhas**"))
    frame = table(name)
    display(frame if not frame.empty else Markdown("Sem dados nesta execução."))

def figure(name):
    path = folder / "figures" / name
    if path.is_file():
        display(Image(filename=str(path)))

## 1. Panorama e limites do corpus

Documentos, registros e estudos revisados são unidades distintas. Registros consolidados por NCT não comprovam identidade clínica. Não somar participantes ou os dois pilotos. Ano de publicação e início do ensaio são variáveis diferentes. Datas posteriores ao corte permanecem sinalizadas para conferência; não são corrigidas por inferência.

In [ ]:
display(pd.DataFrame([{k: summary.get(k) for k in ("documentCount", "publicationCount", "trialCount", "studyCount", "distinctConfirmedStudyCount")}]))
show("corpus_temporal")
show("corpus_temporal_flags")
show("corpus_distributions")
show("corpus_enrollment")

## 2. Patrocinadores e disponibilidade documental

Categorias são as declaradas pelas fontes; nomes não passaram por resolução institucional. Patrocinador não equivale a financiador ou risco de viés. Completude mede disponibilidade, sem pontuação de qualidade metodológica. Numeradores, denominadores e ausências permanecem nas tabelas.

In [ ]:
show("corpus_sponsor_distributions")
show("corpus_sponsor_completeness")
figure("sponsors-types.png")
show("corpus_completeness")

## 3. Relato de resultados

Disponibilidade no registro e publicação vinculada são indicadores diferentes. Os intervalos abaixo são dias de calendário entre campos do registro, entre pares elegíveis segundo o protocolo de auditoria; não representam sobrevida dos pacientes. Não há censura, Kaplan–Meier ou imputação de datas parciais.

In [ ]:
display(pd.DataFrame([summary["reporting"]["interval_summary"]]))
show("reporting_availability")
show("reporting_intervals")
figure("reporting-intervals.png")

## 4. Referências e lacunas bibliográficas

RESULT, DERIVED e BACKGROUND preservam os códigos da fonte. Coincidência por PMID é candidato bibliográfico, sem confirmar vínculo clínico ou suporte semântico. Ausência no corpus não significa inexistência de publicação ou evidência. Tipos podem compartilhar PMIDs. Denominador zero permanece ausente, sem percentual artificial.

In [ ]:
show("publication_reference_coverage")
show("publication_reference_missing")
figure("publication-refs-coverage.png")

## 5. Resposta clínica disponível nesta execução

A síntese usa exclusivamente resultados aprovados e publicados. Um panorama amplo de registros não responde quais medicamentos funcionam. Resultados vazios não autorizam classificar tratamentos como ineficazes nem afirmar ausência de estudos elegíveis. As categorias clínicas dependem de julgamento humano documentado por comparação, população, desfecho e tempo.

In [ ]:
display(pd.DataFrame([summary["clinical"]]))
show("mart_panorama")
show("clinical_results")
show("clinical_synthesis")
if not summary["clinical"]["reviewed_results"]:
    display(Markdown("**Sem resultados clínicos revisados nesta execução. Ainda não é possível concluir quais intervenções beneficiam pacientes com dengue.**"))

## 6. Reprodução e próximos passos

Os módulos Python geram as tabelas; este notebook apenas as consulta. Os hashes abaixo permitem conferir equivalência entre execuções sem depender do estilo visual do HTML. A entrada e o manifesto publicados são imutáveis e verificados. A execução registra seu ambiente em um manifesto separado.

Próximo marco científico: concluir seleção independente, adjudicar divergências, revisar resultados por braço/comparação/população/tempo e avaliar limitações metodológicas. A coleta completa continua sujeita à calibração prevista no protocolo. Não há ranking de medicamentos, meta-análise ou recomendação terapêutica.

In [ ]:
display(pd.DataFrame([
    {"table": name, "rows": meta["rows"], "logical_hash": meta["logical_hash"]}
    for name, meta in sorted(manifest["tables"].items()) if meta["public"]
]))
assert require_published(root, run_id) == manifest, "A execução publicada mudou durante a leitura."
print("Integridade verificada; somente tabelas públicas foram consultadas.")